# Dataset preprocessing

In [4]:
import pandas as pd

def filter_csv(input_file, output_file, chunksize=100000):
    # Open output file in write mode and write the header once
    first_chunk = True
    
    for chunk in pd.read_csv(input_file, chunksize=chunksize):
        if 'LEVEL1ID' not in chunk.columns:
            print("Error: 'LEVEL1ID' column not found.")
            return
        
        # Filter for Health Canada (LEVEL1ID == 6)
        filtered_chunk = chunk[chunk['LEVEL1ID'] == 6]
        
        # Append to output file
        filtered_chunk.to_csv(output_file, mode='a', header=first_chunk, index=False)
        first_chunk = False  # Ensure only the first chunk writes the header
    
    print(f"Filtered data saved to {output_file}")

# Example usage
input_csv = "../data/raw/Main Subset.csv"  
output_csv = "../data/processed/mainsubset.csv"
filter_csv(input_csv, output_csv)


Filtered data saved to ../data/processed/mainsubset.csv


In [8]:
import pandas as pd

# Load both CSV files
df1 = pd.read_csv("../data/raw/Subset 4a.csv", encoding="ISO-8859-1")
df2 = pd.read_csv("../data/raw/Subset 4b.csv", encoding="ISO-8859-1")

# Append the contents
df_combined = pd.concat([df1, df2], ignore_index=True)

# Save to a new CSV file
df_combined.to_csv("Subset 4.csv", index=False)

print("CSV files merged successfully!")


CSV files merged successfully!


In [ ]:
import pandas as pd

def filter_csv(input_file, output_file):
    # Read the CSV file
    df = pd.read_csv(input_file, encoding='ISO-8859-1', low_memory=False)

    print(f"Data Size {df.size}")
    
     # Filter rows where LEVEL1ID == 6 which is Health Canada
    filtered_df = df[(df['LEVEL1ID'] == 6)]
    
    # Write the filtered rows to a new CSV file
    filtered_df.to_csv(output_file, index=False)
    print(f"Filtered data saved to {output_file}")
    
    print(f"Filtered Data Size {filtered_df.size}")

# Process multiple files
for i in range(7, 8):
    input_csv = f"../data/raw/Subset 4.csv"  # Replace with actual input file pattern
    output_csv = f"../data/processed/subset4.csv"  # Desired output file pattern
    filter_csv(input_csv, output_csv)



In [3]:
import pandas as pd

def filter_csv(input_file, output_file):
    # Read the CSV file
    df = pd.read_csv(input_file, encoding='ISO-8859-1', low_memory=False)

    print(f"Data Size {df.size}")
    
     # Filter rows where LEVEL1ID == 6 which is Health Canada
    filtered_df = df[(df['LEVEL1ID'] == 6)]
    
    # Write the filtered rows to a new CSV file
    filtered_df.to_csv(output_file, index=False)
    print(f"Filtered data saved to {output_file}")
    
    print(f"Filtered Data Size {filtered_df.size}")

# Process file
input_csv = f"../data/raw/EEVD.csv"  # input file pattern
output_csv = f"../data/processed/eedv.csv"  # desired output file pattern
filter_csv(input_csv, output_csv)

Data Size 31163090
Filtered data saved to ../data/processed/eedv.csv
Filtered Data Size 1036210


In [27]:
import pandas as pd
import re

# Load the Excel file
file_path = '../data/raw/Lookup.xlsx'
df = pd.read_excel(file_path, sheet_name='QUESTIONS')

# Define a function to clean the question text
def clean_question_text(question):
    # Remove the prefix "Question X. " or similar
    match = re.match(r'Question\s+\d+[a-zA-Z]?\.\s*(.*)', question)
    return match.group(1) if match else question

# Apply the function to the 'English' column to create the cleaned question column
df['Question'] = df['English'].apply(clean_question_text)

# Save the cleaned data to a new Excel file with the required columns and the same sheet name
new_file_path = '../data/processed/lookup.xlsx'
with pd.ExcelWriter(new_file_path, mode='w') as writer:
    df.to_excel(writer, sheet_name='QUESTIONS', index=False, columns=['Question number/numéro de la question', 'Question'])

print("Data saved to a new Excel file with the 'Question number' and 'Question' columns.")


Data saved to a new Excel file with the 'Question number' and 'Question' columns.


In [8]:
import pandas as pd

# Load your data
file_path = '../data/raw/Lookup.xlsx'
df = pd.read_excel(file_path, sheet_name='RESPONSE OPTIONS DE RÉPONSES')

# Function to create QUESTION_TYPE and retain individual columns
def create_question_type_and_retain_columns(df):
    columns_to_group = ['Answer1ENG', 'Answer2ENG', 'Answer3ENG', 'Answer4ENG',
                        'Answer5ENG', 'Answer6ENG', 'Answer7ENG', 'PositiveENG',
                        'NegativeENG', 'AnscountENG']
    
    question_type_mapping = {}
    question_type_counter = 1
    question_type_list = []
    retained_columns_list = []

    for index, row in df.iterrows():
        mapped_values = {col: row[col] for col in columns_to_group if pd.notnull(row[col])}
        question_type_values = ', '.join([str(row[col]) for col in columns_to_group if pd.notnull(row[col])])
        
        if question_type_values not in question_type_mapping:
            question_type_mapping[question_type_values] = question_type_counter
            question_type_counter += 1
        
        question_type = question_type_mapping[question_type_values]
        question_type_list.append((row['QUESTION'], question_type))
        retained_columns_list.append(mapped_values)

    question_type_df = pd.DataFrame(question_type_list, columns=['QUESTION', 'QUESTION_TYPE'])
    retained_columns_df = pd.DataFrame(retained_columns_list)
    
    result_df = pd.concat([question_type_df, retained_columns_df], axis=1)
    return result_df

# Create the new DataFrame
result_df = create_question_type_and_retain_columns(df)

# Save the result to a new CSV file
result_df.to_csv('question_type.csv', index=False)


In [51]:
import pandas as pd

def preprocess_and_check_rows(subset_paths, eedv_path, output_path_non_matching, output_path_matching):
    # Combine all subset files
    subsets_combined = pd.concat([pd.read_csv(path) for path in subset_paths], ignore_index=True)
    eedv = pd.read_csv(eedv_path)
    
    # Align column names and order
    common_columns = subsets_combined.columns.intersection(eedv.columns)
    subsets_combined = subsets_combined[common_columns]
    eedv = eedv[common_columns]

    # Standardize and clean data
    subsets_combined = subsets_combined.fillna('').astype(str)
    eedv = eedv.fillna('').astype(str)

    # Fix: Apply string methods to each string element within the Series
    subsets_combined = subsets_combined.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
    eedv = eedv.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)

    # Convert numeric-like strings to consistent format
    subsets_combined = subsets_combined.applymap(lambda x: int(float(x)) if str(x).replace('.', '', 1).isdigit() else x)
    eedv = eedv.applymap(lambda x: int(float(x)) if str(x).replace('.', '', 1).isdigit() else x)

    # Check row-by-row existence in eedv
    subsets_combined['Exists in EEDV'] = subsets_combined.apply(tuple, axis=1).isin(eedv.apply(tuple, axis=1))

    # Debugging output: count matches and mismatches
    matching_rows_count = subsets_combined['Exists in EEDV'].sum()
    non_matching_rows_count = len(subsets_combined) - matching_rows_count
    
    print(f"Total rows in subsets: {len(subsets_combined)}")
    print(f"Rows found in EEDV: {matching_rows_count}")
    print(f"Rows not found in EEDV: {non_matching_rows_count}")

    # Save rows for further inspection
    subsets_combined[subsets_combined['Exists in EEDV'] == False].to_csv(output_path_non_matching, index=False)
    subsets_combined[subsets_combined['Exists in EEDV'] == True].to_csv(output_path_matching, index=False)

    # Final report
    print(f"Non-matching rows saved to: {output_path_non_matching}")
    print(f"Matching rows saved to: {output_path_matching}")

# Paths to files
subset_paths = [f"../data/processed/subset{i}.csv" for i in range(1, 8)]
eedv_path = "../data/processed/eedv.csv"
output_path_non_matching = "../data/debug/non_matching_rows.csv"
output_path_matching = "../data/debug/matching_rows.csv"

# Call the function
preprocess_and_check_rows(subset_paths, eedv_path, output_path_non_matching, output_path_matching)


C:\Users\Kavya\AppData\Local\Temp\ipykernel_29160\3861837880.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  subsets_combined = subsets_combined.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
C:\Users\Kavya\AppData\Local\Temp\ipykernel_29160\3861837880.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  eedv = eedv.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
C:\Users\Kavya\AppData\Local\Temp\ipykernel_29160\3861837880.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  subsets_combined = subsets_combined.applymap(lambda x: int(float(x)) if str(x).replace('.', '', 1).isdigit() else x)
C:\Users\Kavya\AppData\Local\Temp\ipykernel_29160\3861837880.py:23: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  eedv = eedv.applymap(lambda x: int(float(x)) if str(x).replace('.', '', 1).isdigit() else x)


Total rows in subsets: 106338
Rows found in EEDV: 444
Rows not found in EEDV: 105894
Non-matching rows saved to: ../data/debug/non_matching_rows.csv
Matching rows saved to: ../data/debug/matching_rows.csv
